In [2]:
#ZARA DATASET

# IMPORTAR LIBRERIAS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr
import warnings
warnings.filterwarnings('ignore')
# Configuració estètica
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from scipy.stats import spearmanr, pearsonr, shapiro
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import shapiro
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
import numpy as np
# Configuración de estilo
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Librerías importadas correctamente")


# Si el archivo usa otro separador como punto y coma
df = pd.read_csv(
    r"C:\Users\Lucan Vieira\Documents\GitHub\Proyectos\Zara\Data\zara_cleaned_data 23_03_2026.csv",
    sep=',',  # o el delimitador que corresponda
    engine='python'
)

# Mostrar las primeras filas para verificar
print(df.head())

print("Tipos de datos por columna:")
print(df.dtypes) 
# Información detallada
print("\nInformación completa del dataset:")
print(df.info())

# Estadísticas descriptivas por tipo de dato
print("\nResumen de datos numéricos:")
print(df.describe())

print("\nFirst 5 rows:")
df.head()


✅ Librerías importadas correctamente
   Product ID Product Position Promotion Product Category Seasonal  \
0      185102            Aisle        No         Clothing       No   
1      188771            Aisle        No         Clothing       No   
2      180176          End-cap       Yes         Clothing      Yes   
3      112917            Aisle       Yes         Clothing      Yes   
4      192936          End-cap        No         Clothing      Yes   

   Sales Volume brand                                                url  \
0          2823  Zara  https://www.zara.com/us/en/basic-puffer-jacket...   
1           654  Zara  https://www.zara.com/us/en/tuxedo-jacket-p0889...   
2          2220  Zara  https://www.zara.com/us/en/slim-fit-suit-jacke...   
3          1568  Zara  https://www.zara.com/us/en/stretch-suit-jacket...   
4          2942  Zara  https://www.zara.com/us/en/double-faced-jacket...   

                sku                  name  \
0   272145190-250-2   BASIC PUFFER JACKE

,Product ID,Product Position,Promotion,Product Category,Seasonal,Sales Volume,brand,url,sku,name,description,price,currency,scraped_at,terms,section
0,185102,Aisle,No,Clothing,No,2823,Zara,https://www.zara.com/us/en/basic-puffer-jacket...,272145190-250-2,BASIC PUFFER JACKET,Puffer jacket made of tear-resistant ripstop f...,19.99,USD,2024-02-19 08:50:05.654618,jackets,MAN
1,188771,Aisle,No,Clothing,No,654,Zara,https://www.zara.com/us/en/tuxedo-jacket-p0889...,324052738-800-46,TUXEDO JACKET,Straight fit blazer. Pointed lapel collar and ...,169.00,USD,2024-02-19 08:50:06.590930,jackets,MAN
2,180176,End-cap,Yes,Clothing,Yes,2220,Zara,https://www.zara.com/us/en/slim-fit-suit-jacke...,335342680-800-44,SLIM FIT SUIT JACKET,Slim fit jacket. Notched lapel collar. Long sl...,129.00,USD,2024-02-19 08:50:07.301419,jackets,MAN
3,112917,Aisle,Yes,Clothing,Yes,1568,Zara,https://www.zara.com/us/en/stretch-suit-jacket...,328303236-420-44,STRETCH SUIT JACKET,Slim fit jacket made of viscose blend fabric. ...,129.00,USD,2024-02-19 08:50:07.882922,jackets,MAN
4,192936,End-cap,No,Clothing,Yes,2942,Zara,https://www.zara.com/us/en/double-faced-jacket...,312368260-800-2,DOUBLE FACED JACKET,Jacket made of faux leather faux shearling wit...,139.00,USD,2024-02-19 08:50:08.453847,jackets,MAN


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

# Assuming your original dataframe is called 'df'
# If not, please load it first

# Definir la ruta donde se guardarán los archivos
output_path = r'C:\Users\Lucan Vieira\Documents\GitHub\Proyectos\Zara\Data'

# Crear la carpeta si no existe
if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"📁 Carpeta creada: {output_path}")
else:
    print(f"📁 Usando carpeta existente: {output_path}")

# Create Star Schema

# 1. DIMENSION TABLES

# Dim_Product
dim_product = df[['Product ID', 'name', 'description', 'sku', 'brand', 'url', 'price', 'currency']].copy()
dim_product = dim_product.drop_duplicates(subset=['Product ID']).reset_index(drop=True)
dim_product['Product Key'] = range(1, len(dim_product) + 1)

# Dim_Date (from scraped_at)
dim_date = df[['scraped_at']].copy()
dim_date['scraped_at'] = pd.to_datetime(dim_date['scraped_at'])
dim_date['Date Key'] = dim_date['scraped_at'].dt.strftime('%Y%m%d').astype(int)
dim_date['Year'] = dim_date['scraped_at'].dt.year
dim_date['Month'] = dim_date['scraped_at'].dt.month
dim_date['Day'] = dim_date['scraped_at'].dt.day
dim_date['Quarter'] = dim_date['scraped_at'].dt.quarter
dim_date['Weekday'] = dim_date['scraped_at'].dt.day_name()
dim_date['Month Name'] = dim_date['scraped_at'].dt.month_name()
dim_date = dim_date.drop_duplicates(subset=['Date Key']).reset_index(drop=True)

# Dim_Location (Product Position)
dim_location = df[['Product Position']].copy()
dim_location = dim_location.drop_duplicates().reset_index(drop=True)
dim_location['Location Key'] = range(1, len(dim_location) + 1)

# Dim_Promotion
dim_promotion = df[['Promotion']].copy()
dim_promotion = dim_promotion.drop_duplicates().reset_index(drop=True)
dim_promotion['Promotion Key'] = range(1, len(dim_promotion) + 1)

# Dim_Seasonal
dim_seasonal = df[['Seasonal']].copy()
dim_seasonal = dim_seasonal.drop_duplicates().reset_index(drop=True)
dim_seasonal['Seasonal Key'] = range(1, len(dim_seasonal) + 1)

# Dim_Category
dim_category = df[['Product Category', 'terms', 'section']].copy()
dim_category = dim_category.drop_duplicates(subset=['Product Category', 'terms', 'section']).reset_index(drop=True)
dim_category['Category Key'] = range(1, len(dim_category) + 1)

# 2. FACT TABLE
# Merge all dimension keys to create the fact table

# Create mappings
product_mapping = dim_product[['Product ID', 'Product Key']].copy()
date_mapping = dim_date[['scraped_at', 'Date Key']].copy()
location_mapping = dim_location[['Product Position', 'Location Key']].copy()
promotion_mapping = dim_promotion[['Promotion', 'Promotion Key']].copy()
seasonal_mapping = dim_seasonal[['Seasonal', 'Seasonal Key']].copy()
category_mapping = dim_category[['Product Category', 'terms', 'section', 'Category Key']].copy()

# Prepare fact table
fact_sales = df[['Product ID', 'Sales Volume', 'scraped_at', 'Product Position', 
                  'Promotion', 'Seasonal', 'Product Category', 'terms', 'section']].copy()

# Convert scraped_at to datetime for merging
fact_sales['scraped_at'] = pd.to_datetime(fact_sales['scraped_at'])

# Merge all dimension keys
fact_sales = fact_sales.merge(product_mapping, on='Product ID', how='left')
fact_sales = fact_sales.merge(date_mapping, on='scraped_at', how='left')
fact_sales = fact_sales.merge(location_mapping, on='Product Position', how='left')
fact_sales = fact_sales.merge(promotion_mapping, on='Promotion', how='left')
fact_sales = fact_sales.merge(seasonal_mapping, on='Seasonal', how='left')
fact_sales = fact_sales.merge(category_mapping, on=['Product Category', 'terms', 'section'], how='left')

# Select only the keys and measures for the fact table
fact_sales = fact_sales[['Product Key', 'Date Key', 'Location Key', 'Promotion Key', 
                          'Seasonal Key', 'Category Key', 'Sales Volume']]

# 3. SAVE ALL TABLES AS CSV IN THE SPECIFIED LOCATION

# Save dimension tables
dim_product.to_csv(os.path.join(output_path, 'dim_product.csv'), index=False)
dim_date.to_csv(os.path.join(output_path, 'dim_date.csv'), index=False)
dim_location.to_csv(os.path.join(output_path, 'dim_location.csv'), index=False)
dim_promotion.to_csv(os.path.join(output_path, 'dim_promotion.csv'), index=False)
dim_seasonal.to_csv(os.path.join(output_path, 'dim_seasonal.csv'), index=False)
dim_category.to_csv(os.path.join(output_path, 'dim_category.csv'), index=False)

# Save fact table
fact_sales.to_csv(os.path.join(output_path, 'fact_sales.csv'), index=False)

# 4. DISPLAY SUMMARY

print("=" * 60)
print("STAR SCHEMA CREATED SUCCESSFULLY!")
print("=" * 60)
print(f"\n📁 ARCHIVOS GUARDADOS EN: {output_path}")
print("\n📄 FILES SAVED:")
print("  ✓ dim_product.csv      (Products dimension)")
print("  ✓ dim_date.csv         (Date dimension)")
print("  ✓ dim_location.csv     (Product position dimension)")
print("  ✓ dim_promotion.csv    (Promotion dimension)")
print("  ✓ dim_seasonal.csv     (Seasonal dimension)")
print("  ✓ dim_category.csv     (Category dimension)")
print("  ✓ fact_sales.csv       (Sales fact table)")

print("\n📊 DIMENSION TABLES SUMMARY:")
print(f"  dim_product:     {len(dim_product):3d} records")
print(f"  dim_date:        {len(dim_date):3d} records")
print(f"  dim_location:    {len(dim_location):3d} records")
print(f"  dim_promotion:   {len(dim_promotion):3d} records")
print(f"  dim_seasonal:    {len(dim_seasonal):3d} records")
print(f"  dim_category:    {len(dim_category):3d} records")

print(f"\n📈 FACT TABLE:")
print(f"  fact_sales:      {len(fact_sales):3d} records")
print(f"  Total sales volume: {fact_sales['Sales Volume'].sum():,} units")

# Verificar que los archivos se crearon correctamente
print("\n🔍 VERIFICACIÓN DE ARCHIVOS:")
for file in ['dim_product.csv', 'dim_date.csv', 'dim_location.csv', 
             'dim_promotion.csv', 'dim_seasonal.csv', 'dim_category.csv', 
             'fact_sales.csv']:
    file_path = os.path.join(output_path, file)
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path) / 1024
        print(f"  ✅ {file} ({file_size:.2f} KB)")
    else:
        print(f"  ❌ {file} - No se pudo crear")

print("\n🔍 PREVIEW OF FACT TABLE:")
print(fact_sales.head())

print("\n🔍 SAMPLE OF DIMENSION TABLES:")
print("\ndim_product sample:")
print(dim_product[['Product Key', 'Product ID', 'name', 'price']].head())
print("\ndim_date sample:")
print(dim_date[['Date Key', 'Year', 'Month', 'Month Name', 'Weekday']].head())